# ETS MARL (happo_current) — One-Year Deep Debug

This notebook is set up to inspect one selected year in full detail (decisions, market outcomes, compliance, and budgets).

Default behavior:
- uses your main config (`configs/default.yaml`)
- runs with **no bots** (`N_BOTS = 0`) so only learning-agent behavior is visible
- lets you choose the exact year to inspect (`INSPECT_YEAR`)
- exposes raw policy actions and executed outcomes side by side

Set `CHECKPOINT_DIR` to evaluate learned actor checkpoints instead of heuristic fallback.


In [13]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "configs").exists() and (parent / "src").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.debug_happo_current import (
    load_config,
    configure_simulation,
    build_learning_auction_actions,
    build_learning_secondary_actions,
    build_per_participant_df,
    check_year_constraints,
    load_checkpoint_agents,
    participant_name,
    participant_type,
    BUILDABLE_TECH_NAMES,
)
from src.environment.ets_environment import ETSEnvironment

CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"

# Main controls
SEED = 42
N_YEARS = 12
INSPECT_YEAR = 1  # 1-based
N_LEARNING_AGENTS = 8
N_BOTS = 0  # set to 0 to remove bots from this debug view

# Optional: point to a checkpoint folder with agent_<i>_best.pt files
CHECKPOINT_DIR = None
DETERMINISTIC_POLICY = True

raw_cfg = load_config(CONFIG_PATH)
cfg = configure_simulation(
    raw_cfg,
    n_years=N_YEARS,
    n_learning_agents=N_LEARNING_AGENTS,
    n_bots=N_BOTS,
)

if INSPECT_YEAR < 1 or INSPECT_YEAR > cfg["simulation"]["n_years"]:
    raise ValueError(f"INSPECT_YEAR must be in [1, {cfg['simulation']['n_years']}].")

policy_agents = None
learning_policy_mode = "heuristic"
if CHECKPOINT_DIR is not None:
    checkpoint_path = Path(CHECKPOINT_DIR)
    policy_agents = load_checkpoint_agents(cfg, seed=SEED, checkpoint_dir=checkpoint_path)
    learning_policy_mode = "checkpoint_actor"

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")
print(f"Simulation years: {cfg['simulation']['n_years']} | Inspect year: {INSPECT_YEAR}")
print(f"Participants: learning={cfg['companies']['n_agents']}, bots={cfg['companies']['n_bot_agents']}")
print(f"Learning policy mode: {learning_policy_mode}")


Project root: c:\Users\danie\Documents\Python_Scripts\Master Thesis\Thesis-Energy-Auction\ets_marl_happo_current
Config path: c:\Users\danie\Documents\Python_Scripts\Master Thesis\Thesis-Energy-Auction\ets_marl_happo_current\configs\default.yaml
Simulation years: 12 | Inspect year: 1
Participants: learning=8, bots=0
Learning policy mode: heuristic


In [14]:
env = ETSEnvironment(cfg, seed=SEED)
env.reset(seed=SEED)

year_overview_rows = []
inspect_data = None

for year_idx in range(cfg["simulation"]["n_years"]):
    pre_suspension = env._suspension_remaining.copy()
    pre_cash = np.array(
        [max(0.0, float(c.annual_budget - c.budget_spent_this_year)) for c in env.companies],
        dtype=float,
    )
    carry_start = np.array([float(c._carry_forward) for c in env.companies], dtype=float)
    annual_budget_start = np.array([float(c.annual_budget) for c in env.companies], dtype=float)
    budget_spent_start = np.array([float(c.budget_spent_this_year) for c in env.companies], dtype=float)

    auc_actions, learning_action_source = build_learning_auction_actions(
        env,
        cfg,
        policy_agents=policy_agents,
        deterministic=DETERMINISTIC_POLICY,
    )
    obs_phase2, _ = env.step_auction(auc_actions)

    sec_actions, _ = build_learning_secondary_actions(
        env,
        cfg,
        obs_phase2=obs_phase2,
        policy_agents=policy_agents,
        deterministic=DETERMINISTIC_POLICY,
    )
    _, _, terminated, truncated, info = env.step_secondary(sec_actions)
    log = info["year_log"]

    year_issues = check_year_constraints(env, cfg, log, carry_start, pre_suspension, pre_cash)

    year_overview_rows.append(
        {
            "year": year_idx + 1,
            "learning_action_source": learning_action_source,
            "clearing_price": float(log.get("clearing_price", np.nan)),
            "secondary_clearing": float(log.get("secondary_clearing", np.nan)),
            "auction_volume_mt": float(log.get("auction_volume", np.nan)),
            "allocated_mt": float(log.get("auction_stats", {}).get("total_allocated", np.nan)),
            "defaults": int(log.get("auction_stats", {}).get("defaults", 0)),
            "emissions_total_mt": float(np.sum(log.get("emissions", []))),
            "penalty_total_meur": float(np.sum(log.get("penalties", []))),
            "budget_spent_total_meur": float(sum(c.budget_spent_this_year for c in env.companies)),
            "issues_this_year": int(len(year_issues)),
        }
    )

    if (year_idx + 1) == INSPECT_YEAR:
        raw_action_rows = []
        for i in range(env.n_agents):
            tech_choice = int(np.argmax(auc_actions[i, 3:6]))
            raw_action_rows.append(
                {
                    "participant": participant_name(env, i),
                    "type": participant_type(env, i),
                    "auction_bid_price_raw": float(auc_actions[i, 0]),
                    "auction_qty_mult_raw": float(auc_actions[i, 1]),
                    "auction_invest_frac_raw": float(auc_actions[i, 2]),
                    "auction_tech_choice_raw": BUILDABLE_TECH_NAMES[tech_choice],
                    "secondary_price_raw": float(sec_actions[i, 0]),
                    "secondary_qty_raw": float(sec_actions[i, 1]),
                }
            )

        participant_df = build_per_participant_df(
            env,
            log,
            carry_start,
            learning_action_source=learning_action_source,
        )

        per_agent_diag_rows = []
        for idx_raw, diag in log.get("per_agent_diag", {}).items():
            idx = int(idx_raw)
            row = {
                "participant": participant_name(env, idx),
                "type": participant_type(env, idx),
            }
            row.update(diag)
            per_agent_diag_rows.append(row)

        timeline_rows = [
            {
                "moment": "start_of_year",
                "cap_mt": float(log.get("cap", np.nan)),
                "tnac_mt": float(log.get("tnac", np.nan)),
                "auction_volume_mt": float(log.get("auction_volume", np.nan)),
                "clearing_price": np.nan,
                "secondary_clearing": np.nan,
                "emissions_total_mt": np.nan,
                "shortfall_total_mt": np.nan,
                "penalty_total_meur": np.nan,
                "end_bank_total_mt": np.nan,
            },
            {
                "moment": "post_auction",
                "cap_mt": float(log.get("cap", np.nan)),
                "tnac_mt": float(log.get("tnac", np.nan)),
                "auction_volume_mt": float(log.get("auction_volume", np.nan)),
                "clearing_price": float(log.get("clearing_price", np.nan)),
                "secondary_clearing": np.nan,
                "emissions_total_mt": np.nan,
                "shortfall_total_mt": np.nan,
                "penalty_total_meur": np.nan,
                "end_bank_total_mt": np.nan,
            },
            {
                "moment": "post_secondary_and_compliance",
                "cap_mt": float(log.get("cap", np.nan)),
                "tnac_mt": float(log.get("tnac", np.nan)),
                "auction_volume_mt": float(log.get("auction_volume", np.nan)),
                "clearing_price": float(log.get("clearing_price", np.nan)),
                "secondary_clearing": float(log.get("secondary_clearing", np.nan)),
                "emissions_total_mt": float(np.sum(log.get("emissions", []))),
                "shortfall_total_mt": float(np.sum(log.get("shortfalls", []))),
                "penalty_total_meur": float(np.sum(log.get("penalties", []))),
                "end_bank_total_mt": float(np.sum(log.get("holdings", []))),
            },
        ]

        budget_rows = []
        for i in range(env.n_total):
            budget_rows.append(
                {
                    "participant": participant_name(env, i),
                    "type": participant_type(env, i),
                    "annual_budget_start_meur": float(annual_budget_start[i]),
                    "budget_spent_start_meur": float(budget_spent_start[i]),
                    "payment_meur": float(log["payments"][i]),
                    "trade_cost_meur": float(log["trade_costs"][i]),
                    "invest_cost_meur": float(log["invest_costs"][i]),
                    "mac_cost_meur": float(log.get("mac_costs", [0.0] * env.n_total)[i]),
                    "collateral_cost_meur": float(log.get("collateral_costs", [0.0] * env.n_total)[i]),
                    "penalty_meur": float(log["penalties"][i]),
                    "budget_spent_end_meur": float(env.companies[i].budget_spent_this_year),
                    "available_budget_end_meur": float(env.companies[i].annual_budget - env.companies[i].budget_spent_this_year),
                }
            )

        inspect_data = {
            "timeline_df": pd.DataFrame(timeline_rows),
            "raw_actions_df": pd.DataFrame(raw_action_rows),
            "participant_df": participant_df,
            "budget_df": pd.DataFrame(budget_rows),
            "per_agent_diag_df": pd.DataFrame(per_agent_diag_rows),
            "issues_df": pd.DataFrame(year_issues),
            "learning_action_source": learning_action_source,
            "year_log": log,
        }

    if terminated or truncated:
        break

year_overview_df = pd.DataFrame(year_overview_rows)
if inspect_data is None:
    raise RuntimeError(f"Could not capture INSPECT_YEAR={INSPECT_YEAR}.")


Market calibration: 8 participants, emissions=22.5 Mt, cap=20.3 Mt
[ETSEnvironment] 8 learning + 0 bot = 8 total agents | cancel_under_subscribed=False
[ETSEnvironment] Initial bank seed example (episode-start allowance holdings, unit: MtCO2 allowances): A1=0.10, A2=1.62, A3=0.98, A4=0.92, A5=0.51, A6=0.44, A7=0.20, A8=0.20
[ETSEnvironment] Context: this is each active agent's starting bank before year-1 actions/compliance; total TNAC seed=4.98 MtCO2.


In [15]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

print("Year overview (all simulated years)")
display(year_overview_df)

print(f"Detailed timeline for year {INSPECT_YEAR}")
display(inspect_data["timeline_df"])

print("Learning-agent raw actions (before environment clipping/gates)")
display(inspect_data["raw_actions_df"])

print("Executed per-participant ledger (decision + outcome + compliance)")
display(inspect_data["participant_df"])

print("Budget ledger (start, each cost component, end)")
display(inspect_data["budget_df"])

print("Per-agent diagnostics from environment internals")
if len(inspect_data["per_agent_diag_df"]):
    display(inspect_data["per_agent_diag_df"])
else:
    display(pd.DataFrame([{"note": "No per-agent diagnostics available"}]))

print("Constraint checks for inspected year")
if len(inspect_data["issues_df"]):
    display(inspect_data["issues_df"])
else:
    display(pd.DataFrame([{"note": "No issues detected in inspected year"}]))

print("Available keys in raw year_log (for ad-hoc deep dives)")
display(pd.DataFrame({"year_log_key": sorted(inspect_data["year_log"].keys())}))

# Optional export
# out_dir = PROJECT_ROOT / "results" / "debug_exports"
# out_dir.mkdir(parents=True, exist_ok=True)
# year_overview_df.to_csv(out_dir / "year_overview.csv", index=False)
# inspect_data["timeline_df"].to_csv(out_dir / f"timeline_year_{INSPECT_YEAR}.csv", index=False)
# inspect_data["raw_actions_df"].to_csv(out_dir / f"raw_actions_year_{INSPECT_YEAR}.csv", index=False)
# inspect_data["participant_df"].to_csv(out_dir / f"participant_ledger_year_{INSPECT_YEAR}.csv", index=False)
# inspect_data["budget_df"].to_csv(out_dir / f"budget_ledger_year_{INSPECT_YEAR}.csv", index=False)
# inspect_data["per_agent_diag_df"].to_csv(out_dir / f"per_agent_diag_year_{INSPECT_YEAR}.csv", index=False)
# inspect_data["issues_df"].to_csv(out_dir / f"issues_year_{INSPECT_YEAR}.csv", index=False)
# print(f"Wrote exports to: {out_dir}")


Year overview (all simulated years)


,year,learning_action_source,clearing_price,secondary_clearing,auction_volume_mt,allocated_mt,defaults,emissions_total_mt,penalty_total_meur,budget_spent_total_meur,issues_this_year
0,1,heuristic,108.223976,108.223976,18.251674,18.251674,0,17.569024,474.693613,2772.916623,1
1,2,heuristic,53.595768,103.956579,19.383556,19.383556,0,16.537357,449.205466,1634.587025,0
2,3,heuristic,50.698235,113.566849,16.313096,16.313096,0,16.918373,615.497883,1268.754980,0
3,4,heuristic,127.597137,142.160899,14.801531,14.801531,0,15.608352,901.651325,2309.307403,0
4,5,heuristic,59.615444,128.577947,13.779409,13.779409,0,15.036133,827.444124,1247.673201,0
5,6,heuristic,117.939415,142.463040,12.646802,12.646802,0,15.897848,1015.382132,1906.487641,0
6,7,heuristic,89.614868,135.441547,11.844435,11.844435,0,15.472610,920.564084,1479.241440,0
7,8,heuristic,130.288223,148.112216,11.209522,11.209522,0,17.416176,1126.703491,1882.774555,0
8,9,heuristic,104.927475,143.177817,10.816647,10.816647,0,16.099258,1518.942366,1566.003105,0
9,10,heuristic,76.180916,137.991102,11.493896,11.493896,0,16.040644,1737.495904,1303.708761,0


Detailed timeline for year 1


,moment,cap_mt,tnac_mt,auction_volume_mt,clearing_price,secondary_clearing,emissions_total_mt,shortfall_total_mt,penalty_total_meur,end_bank_total_mt
0,start_of_year,20.2545,4.976398,18.251674,NaN,NaN,NaN,NaN,NaN,NaN
1,post_auction,20.2545,4.976398,18.251674,108.223976,NaN,NaN,NaN,NaN,NaN
2,post_secondary_and_compliance,20.2545,4.976398,18.251674,108.223976,108.223976,17.569024,3.421215,474.693613,9.080262


Learning-agent raw actions (before environment clipping/gates)


...

,participant,type,wtp,wtp_economic,wtp_budget,wtp_binding,bid_price,bid_qty,qty_target,expected_clearing_ma3,actual_clearing,actual_pay,coverage_ratio_post_compliance,marginal_ef,system_ef,revenue,compliance_cost_share_of_budget,invest_frac_pre_compliance_clip,invest_frac_post_compliance_clip,available_budget
0,A1,learning,NaN,243.350696,108.223978,budget,108.223978,5.6920,5.691899,106.768339,108.223976,54.888373,0.000000,0.82,0.270093,1337.950338,0.046311,0.018776,0.018776,986.419286
1,A2,learning,NaN,215.782316,111.954233,budget,111.954233,5.5020,5.502248,106.768339,108.223976,595.448301,0.615419,0.82,0.270093,1337.950338,0.498182,0.024802,0.024802,496.105241
2,A3,learning,NaN,219.400399,147.240475,budget,147.240475,3.8035,3.803302,106.768339,108.223976,411.629886,0.515191,0.82,0.270093,1337.950338,0.367820,0.005000,0.005000,644.227176
3,A4,learning,NaN,219.527187,155.872216,budget,155.872216,3.5925,3.592686,106.768339,108.223976,388.794631,0.631389,0.82,0.270093,1337.950338,0.343034,0.025144,0.025144,583.877836
4,A5,learning,NaN,216.907390,320.495430,economic,216.907390,1.7910,1.790977,106.768339,108.223976,193.829142,0.458065,0.82,0.270093,1337.950338,0.184835,0.005000,0.005000,821.386299
5,A6,learning,NaN,222.039550,299.439896,economic,222.039550,1.9170,1.916912,106.768339,108.223976,207.465368,0.323223,0.82,0.270093,1337.950338,0.198797,0.031272,0.031272,686.780890
6,A7,learning,NaN,214.824597,836.402005,economic,214.824597,0.6530,0.652796,106.768339,108.223976,70.670256,0.750287,0.82,0.270093,1337.950338,0.075066,0.005000,0.005000,844.844548
7,A8,learning,NaN,204.737108,1124.355701,economic,204.737108,0.4855,0.485611,106.768339,108.223976,52.542741,0.542244,0.82,0.270093,1337.950338,0.055203,0.020000,0.020000,781.907264


Constraint checks for inspected year


,type,severity,participant,detail
0,capex_soft_overshoot,soft,A4,capex_spent=138.8742 > capex_limit=130.0000


Available keys in raw year_log (for ad-hoc deep dives)


,year_log_key
0,allocations
1,auction_stats
2,auction_volume
3,bank_start
4,bid_coverages
...,...
63,year
64,year1_tnac
65,year1_tnac_expected_high
66,year1_tnac_expected_low
